# Stage 3 — Micro data

**Friday, 10:50–11:55.**

The same reform, now on SOEP. This notebook is about the pipeline: how prepared survey
data becomes GETTSIM input, what is missing, and how you close the gap.

`soep-preparation` does the basic cleaning: negative codes to missings, dtypes,
variables derived from existing ones, renaming. It does not hand you an analysis-ready
dataset — sample selection and the judgement calls that go with it are still yours. So
what we compute below is testing the waters, not results.

In [ ]:
import sys

sys.path.insert(0, "../src")

import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from gettsim import InputData, MainTarget, TTTargets, copy_environment, main
from gettsim.tt import PiecewisePolynomialParam, TTSIMUnit, get_piecewise_parameters
from soep_preparation.config import BLD

import workshop_soep as ws

POLICY_DATE = ws.POLICY_DATE

## 1 · What the pipeline produces

Cleaned SOEP modules, a metadata catalogue, and `gettsim_inputs`, which renames the
variables it can to GETTSIM qnames. Details in `soep-preparation/docs/scope.md` and
`soep-preparation/docs/using_with_gettsim.md`.

In [ ]:
gettsim_inputs = pd.read_feather(BLD / "gettsim_inputs" / "gettsim_inputs.arrow")
gettsim_inputs.shape

In [ ]:
list(gettsim_inputs.columns)

### The mapping report

The interesting list is the second one: GETTSIM inputs with no SOEP source.

In [ ]:
report = json.loads(
    (BLD / "gettsim_inputs" / "mapping_report.json").read_text(encoding="utf-8")
)["union"]

print(f"GETTSIM inputs in the mapping: {report['n_inputs_total']}")
print(f"  with a SOEP source:          {report['n_inputs_mapped']}")
print(f"  without:                     {report['n_inputs_unmapped']}")

In [ ]:
report["unmapped"][:20]

## 2 · Closing the gap: two trees

`src/workshop_soep.py` holds both, keyed by qname: `from_data` for inputs that come from
SOEP, `ASSUMED` for inputs SOEP does not observe, each with its justification. They are
merged just before the call to `main`.

In [ ]:
ws.assumptions_table()

The derivations in `from_data` are the other half — ordinary pandas, and candidates for
upstreaming into `soep-preparation`'s own mapping.

## 3 · The sample

The 2023 interview under 1 July 2023 law. Wealth is not surveyed in 2023, so the
Vermögensprüfung runs on `soep-preparation`'s imputation projected to 2022, split
equally across household members because GETTSIM's `vermögen` is an individual
input.

`load_soep` drops households with an unobserved rent, heating cost, living space
or income: a missing value reaching GETTSIM is indistinguishable from a zero, and
an unobserved rent would make a household look rent-free. About four in ten go.


In [ ]:
soep = ws.load_soep()
print(f"{len(soep):,} people in {soep['hh_id'].nunique():,} households")
soep[["p_id", "hh_id", "age", "einnahmen__bruttolohn_m", "haushaltsvermögen"]].head()

Households the imputation misses get a wealth of zero, which passes the
Vermögensprüfung — the gap runs towards too much entitlement, not too little.

In [ ]:
ws.wealth_coverage(soep)

In [ ]:
input_tree = ws.input_data_tree(soep)
sorted(input_tree)[:12]

## 4 · Running the reform

Same two environments as yesterday. The cell below is the only one you need to change:
paste your own reform between the markers, or leave the demo reform in place.

In [ ]:
status_quo = main(
    main_target=MainTarget.policy_environment, policy_date_str=POLICY_DATE
)

# ---------------------------- YOUR REFORM GOES HERE ----------------------------
# Paste the reform you built yesterday. If Thursday did not finish, leave this
# cell exactly as it is and work with the demo reform.

reform = copy_environment(status_quo)
reform["bürgergeld"]["parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg"] = (
    PiecewisePolynomialParam(
        value=get_piecewise_parameters(
            func_type="piecewise_linear",
            parameter_list=[
                {"interval": "(-inf, 0)", "intercept": 0, "slope": 0},
                {"interval": "[0, 100)", "slope": 1.0},
                {"interval": "[100, 520)", "slope": 0.2},
                {"interval": "[520, 1000)", "slope": 0.15},
                {"interval": "[1000, 1200)", "slope": 0.1},
                {"interval": "[1200, inf)", "slope": 0.0},
            ],
            leaf_name="parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg",
            xnp=np,
        ),
        input_unit=TTSIMUnit.EUR.PER_MONTH,
        output_unit=TTSIMUnit.EUR.PER_MONTH,
    )
)
# -------------------------------------------------------------------------------

In [ ]:
TARGETS = {
    "bürgergeld": {"betrag_m_bg": None},
    "einkommensteuer": {"betrag_m_sn": None},
    "sozialversicherung": {"beiträge_versicherter_m": None},
    "kindergeld": {"betrag_m": None},
    "wohngeld": {"betrag_m_wthh": None},
}


def run(policy_environment):
    result = main(
        main_target=MainTarget.results.df_with_nested_columns,
        policy_date_str=POLICY_DATE,
        policy_environment=policy_environment,
        input_data=InputData.tree(input_tree),
        tt_targets=TTTargets.tree(TARGETS),
        include_warn_nodes=False,
    )
    result.columns = ["__".join(c).strip("_") for c in result.columns]
    return result.reset_index(drop=True)


before = run(status_quo)
after = run(reform)
before.head()

## 5 · Aggregating at the right level

For the persona every aggregation level coincided. On SOEP they do not: a household can
hold more than one Bedarfsgemeinschaft, more than one Steuernummer. Group-level values
are repeated on every member of the group, so summing them across people double-counts.
Take one value per group, then sum the groups within the household.

In [ ]:
group_ids = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str=POLICY_DATE,
    input_data=InputData.tree(input_tree),
    tt_targets=TTTargets.tree({"bg_id": None, "hh_id": None}),
    include_warn_nodes=False,
).reset_index(drop=True)
group_ids.columns = [
    "__".join(c).strip("_") if isinstance(c, tuple) else c for c in group_ids.columns
]
group_ids.head()

In [ ]:
def bürgergeld_per_household(result):
    """One Bürgergeld amount per Bedarfsgemeinschaft, summed to household level."""
    frame = pd.DataFrame(
        {
            "hh_id": group_ids["hh_id"],
            "bg_id": group_ids["bg_id"],
            "bürgergeld_m": result["bürgergeld__betrag_m_bg"],
        }
    )
    per_bg = frame.drop_duplicates(subset=["bg_id"])
    return per_bg.groupby("hh_id")["bürgergeld_m"].sum()


bürgergeld_before = bürgergeld_per_household(before)
bürgergeld_after = bürgergeld_per_household(after)

## 6 · Weighting

SOEP is a sample. Without the household weight, sums describe the sample, not
Germany. The weights are not rescaled after the dropped households, so the level
below is scaled down with them; shares are unaffected.


In [ ]:
weights = (
    soep.drop_duplicates("hh_id")
    .set_index("hh_id")["hh_weighting_factor"]
    .astype("float64")
)

print(f"households represented: {weights.sum():,.0f}")

In [ ]:
change = (bürgergeld_after - bürgergeld_before).reindex(weights.index).fillna(0.0)

annual_cost = (change * weights).sum() * 12
print(f"fiscal effect: {annual_cost / 1e9:,.2f} bn € per year")

## 7 · Winners and losers

In [ ]:
buckets = pd.cut(
    change,
    bins=[-np.inf, -0.01, 0.01, np.inf],
    labels=["loses", "unchanged", "gains"],
)
shares = weights.groupby(buckets, observed=False).sum() / weights.sum()
shares.map("{:.1%}".format)

## 8 · By income decile

In [ ]:
income = (
    pd.DataFrame(
        {
            "hh_id": group_ids["hh_id"],
            "gross_m": input_tree["einnahmen"]["bruttolohn_m"],
        }
    )
    .groupby("hh_id")["gross_m"]
    .sum()
    .reindex(weights.index)
    .fillna(0.0)
)

decile = pd.qcut(income.rank(method="first"), 10, labels=range(1, 11))
by_decile = (
    pd.DataFrame({"change": change, "weight": weights, "decile": decile})
    .groupby("decile", observed=False)
    .apply(
        lambda g: (g["change"] * g["weight"]).sum() / g["weight"].sum(),
        include_groups=False,
    )
)

fig = go.Figure(
    go.Bar(
        x=by_decile.index.astype(str), y=by_decile.to_numpy(), marker_color="#d62728"
    )
)
fig.update_layout(
    title="Mean change in Bürgergeld by gross-earnings decile",
    xaxis_title="decile of household gross earnings",
    yaxis_title="€ per month",
    template="plotly_white",
    height=420,
)
fig